## PRACTICE EXERCISE — "Broken Deploy" Investigation

**Setup (run this once — it builds a self-contained fake incident in `/tmp/incident`, touches nothing real on your system):**

In [ ]:
mkdir -p /tmp/incident/releases/{v1,v2,v3}
touch /tmp/incident/releases/v1/app.bin /tmp/incident/releases/v2/app.bin /tmp/incident/releases/v3/app.bin
ln -s /tmp/incident/releases/v4 /tmp/incident/current

cat > /tmp/incident/deploy.sh << 'EOF'
#!/bin/bash
echo "deploying..."
EOF
chmod 644 /tmp/incident/deploy.sh

cat > /tmp/incident/config.yaml << 'EOF'
db_password: hunter2
EOF
chmod 644 /tmp/incident/config.yaml

cat > /tmp/incident/app.log << 'EOF'
2026-08-25 10:00:01 INFO Starting app
2026-08-25 10:00:02 ERROR Connection refused to db
2026-08-25 10:00:05 WARN Retrying connection
2026-08-25 10:00:06 ERROR Connection refused to db
2026-08-25 10:00:10 ERROR Timeout waiting for upstream
2026-08-25 10:00:12 ERROR Connection refused to db
2026-08-25 10:00:15 INFO Retry succeeded
2026-08-25 10:00:20 ERROR Connection refused to db
2026-08-25 10:00:25 WARN Slow response 3200ms
2026-08-25 10:00:30 ERROR Connection refused to db
2026-08-25 10:00:35 ERROR Disk write failed
EOF

echo "Incident environment ready in /tmp/incident"

```shell
ln -s /tmp/incident/releases/v4 /tmp/incident/current
       ^^^^^^^^^^^^^^^^^^^^^^^^^ ^^^^^^^^^^^^^^^^^^^^^
       TARGET (what it points TO)   LINK_NAME (the symlink file itself, created)
```
- `/tmp/incident/current` is the **actual symlink file** that gets created.
- `/tmp/incident/releases/v4` is just a **path string stored inside it** — nothing more. Your own `ls -l` output confirms this exactly:

In [ ]:
lrwxrwxrwx 1 dennie dennie 25 ... /tmp/incident/current -> /tmp/incident/releases/v4

Read that arrow the same way you'd read the syntax: `LINK_NAME -> TARGET`. `current` is the symlink; `releases/v4` is what it claims to point at.

`ln -s` does exactly one thing: create a symlink **file** containing a target path string. It never checks whether that target exists, and it never creates it. That's actually a fundamental, deliberate property of symlinks (unlike hardlinks, which _must_ point at something real, since they reference an actual inode) — **a symlink is allowed to point at nothing.** That's precisely why your setup script could create a symlink to `v4` even though only `v1`, `v2`, `v3` were ever `mkdir`'d — nothing stopped it, because `ln -s` doesn't validate.

---

**COMMAND:** `cat > file << 'EOF' ... EOF`

**PURPOSE (support):** writes multi-line text into a file directly from your terminal/script, without needing an external editor — very common in setup scripts, config generation, and Ansible/deploy tooling.

**Breaking it into its 3 pieces:**

1. **`cat > /tmp/incident/deploy.sh`** — `cat` normally reads stdin and prints it to stdout; the `>` redirects that stdout _into_ the file instead of your terminal, **overwriting** it (use `>>` instead of `>` if you wanted to append rather than overwrite — same distinction you already know from your existing redirect knowledge).
    
2. **`<< 'EOF'`** — this is the **here document** (heredoc). It tells bash: "don't wait for interactive input — instead, treat every line I type from here until you see a line that's exactly `EOF` as the stdin content." `EOF` isn't a keyword — it's just a delimiter _I chose_; it could be any word (`END`, `DONE`, whatever), as long as the opening and closing markers match exactly.
    
3. **The closing `EOF`** on its own line — this is what tells bash the heredoc content is finished. Everything between the two `EOF` markers gets fed to `cat`, which (thanks to step 1's redirect) writes it straight into `deploy.sh`.
    

**So the whole block means:** _"Create `deploy.sh` containing exactly these two lines: `#!/bin/bash` and `echo "deploying..."`."_ Equivalent to opening the file in an editor, typing those two lines, and saving — just scriptable and non-interactive.

**GOTCHA worth knowing now:** notice I wrote `<< 'EOF'` with **quotes** around `EOF`, not bare `<< EOF`. That's deliberate — quoting the delimiter disables variable expansion inside the block. Compare:

In [ ]:
dennie@HP:~/linux-tut$ cat > /tmp/test1.txt << EOF
> Home dir is $HOME
> EOF
dennie@HP:~/linux-tut$ cat > /tmp/test2.txt << 'EOF'
> Home dir is $HOME
> EOF

In [ ]:
dennie@HP:~/linux-tut$ cat /tmp/test1.txt

Home dir is /home/dennie

In [ ]:
dennie@HP:~/linux-tut$ cat /tmp/test2.txt
Home dir is $HOME

`test1.txt` will show your actual home path (`/home/dennie`) because unquoted `EOF` lets `$HOME` expand; `test2.txt` will literally print `$HOME` as text, because the quotes told bash "treat this heredoc as literal, don't substitute anything." That's exactly why I quoted `'EOF'` in the practice setup — the script content (`#!/bin/bash`, `db_password: hunter2`) needed to land in the file _exactly as written_, with zero shell interpretation.

---

**The situation:** a deploy under `/tmp/incident` is broken in three separate ways, and the app's log has a recurring error you need to identify. Find and fix all of it using only tools from Pillars 1–4.

**Task 1 — find the broken symlink.** Somewhere under `/tmp/incident`, one entry is a symlink pointing at a release that doesn't exist. _Hint: `find -type l`, then confirm it's broken with `readlink -f` or a colored `ls -l`._

**Task 2 — the deploy script won't run.** Try executing `/tmp/incident/deploy.sh` directly. Diagnose why, then fix it.

**Task 3 — a secrets file has the wrong permissions.** `/tmp/incident/config.yaml` contains a password but is currently readable by anyone on the box. Find it and lock it down to owner-only.

**Task 4 — find the most common error.** Using the pipeline pattern from the primer above, find which single error message appears most often in `/tmp/incident/app.log`, and how many times.

**Task 5 — quick counts.** How many `ERROR` lines total vs. `WARN` lines total are in the log?

---

**VERIFY YOUR WORK — expected results:**

1. `find /tmp/incident -type l` should show exactly one symlink: `/tmp/incident/current`, and it should point at a `v4` release directory that doesn't exist.
2. Before your fix, `ls -l deploy.sh` shows no `x` anywhere; running `./deploy.sh` fails with `Permission denied`. After `chmod +x`, it should print `deploying...`.
3. `config.yaml` starts at `644` (world-readable); after your fix it should read `-rw-------` (`600`).
4. The top error should be `Connection refused to db`, appearing **5** times.
5. `ERROR` count: **7**. `WARN` count: **2**.

---

#### Task 1 — find the broken symlink.

1. Find symlink in `/tmp/incident`:

In [ ]:
dennie@HP:~/linux-tut$ find /tmp/incident/ -type l
/tmp/incident/current

2. Read link with `readlink -f`:

In [ ]:
dennie@HP:~/linux-tut$ readlink -f /tmp/incident/current
/tmp/incident/releases/v4

**`readlink -f`:** notice it happily printed `/tmp/incident/releases/v4` even though that path doesn't exist. `readlink -f` **resolves the path textually** (follows/expands symlinks in the chain) — it does _not_ verify the final target is real. To check existence, you need a separate check like the `ls -ld` above, or `test -e /tmp/incident/releases/v4 && echo exists || echo missing`.

Other options:

In [ ]:
# Option 2: bridge stdin -> arguments with xargs
dennie@HP:~/linux-tut$ find /tmp/incident/ -type l | xargs readlink -f
/tmp/incident/releases/v4

# Option 3: find's own built-in "-exec" (runs a command per match)
dennie@HP:~/linux-tut$ find /tmp/incident -type l -exec readlink -f {} \;
/tmp/incident/releases/v4

**READ THE OUTPUT (option 3):** `{}` is a placeholder `find` substitutes with each matched path, and `\;` marks the end of the command being executed — so this reads as "for each symlink found, run `readlink -f` on it."

3. Confirm the file that the symling pointing to exists or not:

In [ ]:
dennie@HP:~/linux-tut$ ls -l /tmp/incident/releases/
total 12
drwxr-xr-x 2 dennie dennie 4096 Aug 28 04:22 v1
drwxr-xr-x 2 dennie dennie 4096 Aug 28 04:22 v2
drwxr-xr-x 2 dennie dennie 4096 Aug 28 04:22 v3

In [ ]:
dennie@HP:~/linux-tut$ ls -ld /tmp/incident/releases/v4
ls: cannot access '/tmp/incident/releases/v4': No such file or directory

4. There is no `/tmp/incident/releases/v4`, meaning the symlink is broken referencing **non-existent** destination.

---

**Why I built the exercise this exact way — real-world tie-in:** `current -> releases/vN` is the actual, standard deployment pattern used in production ops (often called the "current symlink" or blue-green pattern). Deploying a new version = create `releases/v5`, then atomically repoint `current` at it with `ln -sfn`. Rolling back = repoint it at the old version. The entire "app" just always reads through `current`, oblivious to version numbers. A **broken** `current` symlink — exactly what you're diagnosing — is one of the single most common real "the app won't start after deploy" tickets in the wild.

Now that it's confirmed broken, fix it for real:

In [ ]:
dennie@HP:~/linux-tut$ ln -sfn /tmp/incident/releases/v3 /tmp/incident/current
dennie@HP:~/linux-tut$ ls -l /tmp/incident/current
lrwxrwxrwx 1 dennie dennie 25 Aug 28 05:09 /tmp/incident/current -> /tmp/incident/releases/v3

> `-f` force-overwrites the existing symlink, `-n` treats the link name as a file not a directory to descend into — both matter when repointing an existing symlink.

Confirm it now resolves to a directory that actually exists.

---

#### Task 2 — the deploy script won't run

1. Try to run the script:

In [ ]:
dennie@HP:~/linux-tut$ /tmp/incident/deploy.sh
bash: /tmp/incident/deploy.sh: Permission denied

Permission denied, so the script doesn't execute.

2. Check its permission:

In [ ]:
dennie@HP:~/linux-tut$ ls -l /tmp/incident/deploy.sh 
-rw-r--r-- 1 dennie dennie 32 Aug 29 14:15 /tmp/incident/deploy.sh

There is no execution allowed at all, no `x` anywhere in the string.

3. The fix is to add execution (`x`) to the owner only with `chmode`:

In [ ]:
dennie@HP:~/linux-tut$ chmod u+x /tmp/incident/deploy.sh
dennie@HP:~/linux-tut$ ls -l /tmp/incident/deploy.sh 
-rwxr--r-- 1 dennie dennie 32 Aug 29 14:15 /tmp/incident/deploy.sh

4. Verify the script is now executable by the user:

In [ ]:
dennie@HP:~/linux-tut$ /tmp/incident/deploy.sh
deploying...

**Permission denied → `ls -l` → no `x` anywhere → `chmod u+x` → verified it runs.** That's the exact diagnosis path for the "script won't execute" ticket, done right.

One thing worth calling out specifically: you used `chmod u+x` (owner-only) instead of the more commonly-typed `chmod +x` (which would add execute for **everyone** — owner, group, and other). That's the better call here, and it's the least-privilege instinct we built in Pillar 3 with `sudo` — this script has no reason to be executable by anyone but you, so don't grant more than what's needed just because it's the shorter thing to type. Good judgment.

---

#### Task 3 — a secrets file has the wrong permissions

1. Find the file:

In [ ]:
dennie@HP:~/linux-tut$ ls -l /tmp/incident/config.yaml
-rw-r--r-- 1 dennie dennie 21 Aug 29 14:15 /tmp/incident/config.yaml

dennie@HP:~/linux-tut$ cat /tmp/incident/config.yaml
db_password: hunter2

It contains a password, but is readable by onyone.

2. Fix it, with locking it except the owner:

In [ ]:
dennie@HP:~/linux-tut$ chmod go-r /tmp/incident/config.yaml

3. Verify permissions:

In [ ]:
dennie@HP:~/linux-tut$ ls -l /tmp/incident/config.yaml
-rw------- 1 dennie dennie 21 Aug 29 14:15 /tmp/incident/config.yaml

`chmod go-r` (remove read from group and other) → `-rw-------` (600) — exactly the target state, just reached via the symbolic/relative form instead of numeric `600`. Both are valid; yours is actually a nice illustration of _why_ symbolic mode exists: "strip read from group+other" is more self-documenting here than a bare `600` if someone's reading your history later and doesn't want to mentally decode octal.

---

#### Task 4 — find the most common error
1. Find lines with "error or fail" in them:

In [ ]:
dennie@HP:~/linux-tut$ grep -i "error\|fail" /tmp/incident/app.log
2026-08-25 10:00:02 ERROR Connection refused to db
2026-08-25 10:00:06 ERROR Connection refused to db
2026-08-25 10:00:10 ERROR Timeout waiting for upstream
2026-08-25 10:00:12 ERROR Connection refused to db
2026-08-25 10:00:20 ERROR Connection refused to db
2026-08-25 10:00:30 ERROR Connection refused to db
2026-08-25 10:00:35 ERROR Disk write failed

Look through the most appeared one - `"Connection refused to db"`

2. Count them with `wc -l`

**Option 1**:

In [ ]:
dennie@HP:~/linux-tut$ grep -i "Connection refused to db" /tmp/inci
dent/app.log | wc -l
5

`Connection refused to db`, count **5** — matches. Your approach (eyeball the filtered list, then `grep` that exact string piped to `wc -l` to confirm the count) works and gives the right answer.

> One scaling note worth flagging: this worked because the log was only 11 lines — easy to eyeball. On a real production log with thousands or millions of lines, you can't visually spot "the most common one".

**Option 2**:

That's exactly why the full `sort | uniq -c | sort -rn` pipeline from the lesson exists:

In [ ]:
dennie@HP:~/linux-tut$ grep -i "Connection refused to db" /tmp/incident/app.log | sort | uniq -c | sort -rn
      1 2026-08-25 10:00:30 ERROR Connection refused to db
      1 2026-08-25 10:00:20 ERROR Connection refused to db
      1 2026-08-25 10:00:12 ERROR Connection refused to db
      1 2026-08-25 10:00:06 ERROR Connection refused to db
      1 2026-08-25 10:00:02 ERROR Connection refused to db

it finds the **top message algorithmically**, without you needing to already know what to look for.

**Option 3**:

Worth trying it here too, just to see it arrive at the same answer automatically:

In [ ]:
dennie@HP:~/linux-tut$ grep -i "error\|fail" /tmp/incident/app.log | cut -d' ' -f4- | sort | uniq -c | sort -rn | head -3
      5 Connection refused to db
      1 Timeout waiting for upstream
      1 Disk write failed

(`cut -d' ' -f4-` strips the date/time/level columns, keeping just the message — same idea as the `awk` version from before, simpler syntax since this log's spacing is consistent.) You should see `5 Connection refused to db` land right at the top, with the others below it — proving both methods agree, but only one of them still works at production scale.

---

#### Task 5 — quick counts

1. Locate all errors and warnings:

In [ ]:
dennie@HP:~/linux-tut$ grep -i "error\|fail\|warn" /tmp/incident/app.log
2026-08-25 10:00:02 ERROR Connection refused to db
2026-08-25 10:00:05 WARN Retrying connection
2026-08-25 10:00:06 ERROR Connection refused to db
2026-08-25 10:00:10 ERROR Timeout waiting for upstream
2026-08-25 10:00:12 ERROR Connection refused to db
2026-08-25 10:00:20 ERROR Connection refused to db
2026-08-25 10:00:25 WARN Slow response 3200ms
2026-08-25 10:00:30 ERROR Connection refused to db
2026-08-25 10:00:35 ERROR Disk write failed

You already know exactly what you're counting: `ERROR` lines and `WARN` lines. No filtering/extracting/sorting needed at all.

2. Skip the pipeline entirely and go straight to the original file:

In [ ]:
dennie@HP:~/linux-tut$ grep -c "ERROR" /tmp/incident/app.log 
7
dennie@HP:~/linux-tut$ grep -c "WARN" /tmp/incident/app.log 
2

> `-c` (from the flags table) makes `grep` print just the count of matching lines instead of the lines themselves — that's the whole task in two commands, zero piping.

Using pipline. Group and sort by 'error' and 'warn':

In [ ]:
dennie@HP:~/linux-tut$ grep -i "error\|fail\|warn" /tmp/incident/app.log | cut -d' ' -f3- | sort
ERROR Connection refused to db
ERROR Connection refused to db
ERROR Connection refused to db
ERROR Connection refused to db
ERROR Connection refused to db
ERROR Disk write failed
ERROR Timeout waiting for upstream
WARN Retrying connection
WARN Slow response 3200ms

3. Count errors and warnings:

In [ ]:
dennie@HP:~/linux-tut$ awk '{count[$3]++} END {for (error in count) print error, count[error]}' /tmp/incident/app.log | grep -i "error\|warn"
WARN 2
ERROR 7

**Task 5 confirmed correct — and this is a genuinely nice technique.** `WARN 2`, `ERROR 7` — exact match on both counts. But the method deserves a closer look, because you used something I didn't explicitly teach:

In [ ]:
awk '{count[$3]++} END {for (error in count) print error, count[error]}'

This is `awk`'s **associative array** pattern — `count[$3]++` uses whatever's in column 3 (the log level: `INFO`/`WARN`/`ERROR`) as a _key_ and increments a counter for it, once per line, across the whole file. The `END {...}` block runs exactly once, after every line has been processed, and loops over every key that ended up in the array (`for (error in count)`), printing each one with its final tally.

**Developer bridge:** this is precisely a Python `Counter` or `defaultdict(int)`, written in `awk`'s syntax — `count[$3] += 1` for every line is the exact same idea as `counter[level] += 1` in a Python loop, and the `END` block is your `for key, val in counter.items(): print(key, val)` afterward. You reached for this instinctively, which tells me the mental model transferred cleanly. This is genuinely a better technique than `grep | cut | sort | uniq -c` for this specific job — one pass over the file instead of five piped stages — worth keeping in your toolbox.

**All 5 tasks solved correctly.** Broken symlink diagnosed and fixed, script permission bug diagnosed and fixed with the least-privilege instinct, secrets file locked down, top error identified two different ways, and a clean per-level count using a technique above what was taught. That's the whole of Pillars 1–4 exercised end to end. Nicely done.